# Logistic Regression for activity completion

This notebook trains an explainable model that estimates the probability that an employee will complete a voluntary activity. It uses the fixed temporal split from `activity_completion_logistic.csv`: older rows for training and newer rows for validation.

Identifiers and outcome-time fields are not used as model features. The notebook compares the model with a constant-probability baseline and saves the fitted pipeline, metrics, coefficients, and validation predictions.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = Path.cwd()
if not (DATA_DIR / "activity_completion_logistic.csv").exists():
    candidate = DATA_DIR / "case_1" / "career_quest_dataset"
    if (candidate / "activity_completion_logistic.csv").exists():
        DATA_DIR = candidate
    else:
        raise FileNotFoundError("Run the notebook from the dataset folder or project root.")

DATA_PATH = DATA_DIR / "activity_completion_logistic.csv"
MODEL_PATH = DATA_DIR / "activity_completion_logistic.joblib"
METRICS_PATH = DATA_DIR / "activity_completion_logistic_metrics.json"
COEFFICIENTS_PATH = DATA_DIR / "activity_completion_logistic_coefficients.csv"
PREDICTIONS_PATH = DATA_DIR / "activity_completion_logistic_validation_predictions.csv"

print(f"Dataset folder: {DATA_DIR.resolve()}")

Dataset folder: /Users/alikhan/Documents/HAckAlem/case_1/career_quest_dataset


In [2]:
data = pd.read_csv(DATA_PATH, parse_dates=["date"])

categorical_features = [
    "employee_role",
    "employee_grade",
    "employee_work_format",
    "employee_preferred_language",
    "event_type",
    "event_format",
    "assigned_by",
]
numeric_features = [
    "duration_hours",
    "tenure_days_at_activity",
    "attempt_number",
    "prior_same_event_attempts",
    "employee_prior_outcomes",
    "employee_completion_rate_before",
    "employee_dropped_rate_before",
    "employee_no_show_rate_before",
    "employee_declined_rate_before",
    "employee_format_prior_outcomes",
    "employee_format_completion_rate_before",
    "employee_type_prior_outcomes",
    "employee_type_completion_rate_before",
    "event_prior_outcomes",
    "event_completion_rate_before",
    "is_first_employee_activity",
    "days_since_previous_activity",
]
feature_columns = categorical_features + numeric_features
target_column = "completed_label"

required_columns = set(feature_columns + [target_column, "dataset_split"])
missing_columns = sorted(required_columns - set(data.columns))
assert not missing_columns, f"Missing columns: {missing_columns}"
assert data[feature_columns + [target_column]].isna().sum().sum() == 0
assert set(data["dataset_split"]) == {"train", "validation"}

train = data[data["dataset_split"] == "train"].copy()
validation = data[data["dataset_split"] == "validation"].copy()
X_train, y_train = train[feature_columns], train[target_column].astype(int)
X_validation, y_validation = validation[feature_columns], validation[target_column].astype(int)

print(f"Train: {len(train)} rows, positive rate {y_train.mean():.3f}")
print(f"Validation: {len(validation)} rows, positive rate {y_validation.mean():.3f}")
print(f"Train dates: {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Validation dates: {validation['date'].min().date()} to {validation['date'].max().date()}")

Train: 1178 rows, positive rate 0.701
Validation: 325 rows, positive rate 0.671
Train dates: 2024-10-01 to 2026-04-30
Validation dates: 2026-05-01 to 2026-09-30


In [3]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numeric", numeric_pipeline, numeric_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="lbfgs",
        max_iter=3000,
        random_state=42,
    )),
])
model.fit(X_train, y_train)
train_probability = model.predict_proba(X_train)[:, 1]
validation_probability = model.predict_proba(X_validation)[:, 1]
validation_prediction = (validation_probability >= 0.5).astype(int)
print("Model fitted.")

Model fitted.


In [4]:
def evaluate(y_true, probability, threshold=0.5):
    prediction = (probability >= threshold).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "pr_auc": float(average_precision_score(y_true, probability)),
        "log_loss": float(log_loss(y_true, probability)),
        "brier_score": float(brier_score_loss(y_true, probability)),
        "accuracy": float(accuracy_score(y_true, prediction)),
        "precision": float(precision_score(y_true, prediction, zero_division=0)),
        "recall": float(recall_score(y_true, prediction, zero_division=0)),
        "f1": float(f1_score(y_true, prediction, zero_division=0)),
        "threshold": float(threshold),
        "confusion_matrix": confusion_matrix(y_true, prediction, labels=[0, 1]).tolist(),
    }

baseline_probability = np.full(len(y_validation), y_train.mean(), dtype=float)
metrics = {
    "train_model": evaluate(y_train, train_probability),
    "model": evaluate(y_validation, validation_probability),
    "baseline": evaluate(y_validation, baseline_probability),
    "data": {
        "train_rows": int(len(train)),
        "validation_rows": int(len(validation)),
        "train_positive_rate": float(y_train.mean()),
        "validation_positive_rate": float(y_validation.mean()),
        "validation_start": str(validation["date"].min().date()),
    },
}

comparison = pd.DataFrame([metrics["model"], metrics["baseline"]], index=["Logistic Regression", "Constant baseline"])
comparison.drop(columns=["confusion_matrix"])

,roc_auc,pr_auc,log_loss,brier_score,accuracy,precision,recall,f1,threshold
Logistic Regression,0.65352,0.774390,0.617039,0.211275,0.689231,0.694352,0.958716,0.805395,0.5
Constant baseline,0.50000,0.670769,0.635800,0.221763,0.670769,0.670769,1.000000,0.802947,0.5


In [5]:
confusion = pd.DataFrame(
    metrics["model"]["confusion_matrix"],
    index=["Actual negative", "Actual completed"],
    columns=["Predicted negative", "Predicted completed"],
)
confusion

,Predicted negative,Predicted completed
Actual negative,15,92
Actual completed,9,209


In [6]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": model.named_steps["classifier"].coef_[0],
})
coefficients["odds_ratio"] = np.exp(coefficients["coefficient"])
coefficients["absolute_coefficient"] = coefficients["coefficient"].abs()
coefficients = coefficients.sort_values("absolute_coefficient", ascending=False).reset_index(drop=True)
coefficients.head(20)[["feature", "coefficient", "odds_ratio"]]

,feature,coefficient,odds_ratio
0,categorical__assigned_by_self,0.881416,2.414316
1,categorical__event_type_meetup,-0.384596,0.680726
2,categorical__assigned_by_hr,-0.375545,0.686915
3,categorical__employee_role_Sales Manager,0.375427,1.455612
4,categorical__employee_role_Backend Engineer,-0.325995,0.721809
5,categorical__event_type_certification,0.323693,1.382223
6,categorical__assigned_by_manager,-0.322902,0.724045
7,numeric__duration_hours,-0.276448,0.758473
8,categorical__event_format_self_paced,0.239122,1.270134
9,categorical__employee_role_Customer Support Sp...,0.235937,1.266095


In [7]:
validation_predictions = validation[["record_id", "employee_id", "event_id", "date", target_column]].copy()
validation_predictions["predicted_completion_probability"] = validation_probability
validation_predictions["predicted_label_at_0_5"] = validation_prediction

joblib.dump(model, MODEL_PATH)
with METRICS_PATH.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
    file.write("\n")
coefficients.to_csv(COEFFICIENTS_PATH, index=False)
validation_predictions.to_csv(PREDICTIONS_PATH, index=False, date_format="%Y-%m-%d")

print(f"Saved model: {MODEL_PATH.resolve()}")
print(f"Saved metrics: {METRICS_PATH.resolve()}")
print(f"Saved coefficients: {COEFFICIENTS_PATH.resolve()}")
print(f"Saved validation predictions: {PREDICTIONS_PATH.resolve()}")

Saved model: /Users/alikhan/Documents/HAckAlem/case_1/career_quest_dataset/activity_completion_logistic.joblib
Saved metrics: /Users/alikhan/Documents/HAckAlem/case_1/career_quest_dataset/activity_completion_logistic_metrics.json
Saved coefficients: /Users/alikhan/Documents/HAckAlem/case_1/career_quest_dataset/activity_completion_logistic_coefficients.csv
Saved validation predictions: /Users/alikhan/Documents/HAckAlem/case_1/career_quest_dataset/activity_completion_logistic_validation_predictions.csv
